# 🫁 CoughTB — Training Notebook

Train **MobileNetV4 + Res2TSM** on **CODA TB DREAM Challenge** dataset

## Training Strategy
| Phase | Epochs | Frozen parts | Learning Rate |
|:-----:|:------:|:-------------|:--------------|
| 1 | 10 | Backbone (MobileNetV4) | 1e-3 (head + Res2TSM) |
| 2 | 40 | Nothing (full fine-tune) | 1e-4 (cosine annealing) |

**Optimizer**: AdamW  
**Augmentation**: SpecAugment (time/freq masking), Gaussian noise  
**Batch size**: 32  
**Early stopping**: patience 7 (Phase 2)  
**Split**: Patient-level 80/20 stratified by tb_status

---
## How to Use
1. **First** run `notebooks/precompute_mels.ipynb` to pre-compute mel spectrograms (~10× faster training)
2. Upload the pre-computed ZIP when prompted
3. Wait for training to complete (~2-4 hours on Colab GPU)
4. Download the trained model weights

*No pre-computed data? Cell 2b provides a raw WAV fallback (slower).*

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchaudio timm librosa soundfile numpy pandas matplotlib seaborn scikit-learn scipy pillow tqdm

## 2. Upload Pre-computed Mel Spectrograms (Fast Path) 🚀

**Recommended**: Upload the pre-computed `.npy` files from `notebooks/precompute_mels.ipynb`.
This skips expensive audio → mel conversion during training (~10× faster data loading).

Alternatively, upload raw WAV dataset (see Cell 2b below — slow path).

Your ZIP should contain:
- `precomputed_metadata.csv` — mapping of file → label
- `precomputed_mels/*.npy` — (3, 224, 224) float32 arrays

In [ ]:
import os
import zipfile
import glob
from pathlib import Path
from google.colab import files

print('=' * 60)
print('Upload pre-computed mel spectrograms ZIP')
print('(from notebooks/precompute_mels.ipynb)')
print('=' * 60)
uploaded = files.upload()

# Extract
for fn in uploaded.keys():
    print(f'\nExtracting {fn}...')
    with zipfile.ZipFile(fn, 'r') as zf:
        zf.extractall('precomputed')

# Find the CSV and mel directory
csv_path = None
mels_dir = None

csv_candidates = glob.glob('precomputed/**/*.csv', recursive=True)
for c in csv_candidates:
    if 'metadata' in c.lower():
        csv_path = c
        break
if not csv_path and csv_candidates:
    csv_path = csv_candidates[0]

npy_dirs = list(Path('precomputed').rglob('precomputed_mels'))
if npy_dirs:
    mels_dir = npy_dirs[0]

if csv_path is None or mels_dir is None:
    print('\n❌ Could not find precomputed files. Will try raw dataset path...')
    USE_PRECOMPUTED = False
else:
    USE_PRECOMPUTED = True
    npy_count = len(list(Path(mels_dir).glob('*.npy')))
    print(f'\n✅ Pre-computed data found!')
    print(f'   CSV: {csv_path}')
    print(f'   Mels dir: {mels_dir} ({npy_count} files)')

print(f'\nUsing pre-computed mode: {USE_PRECOMPUTED}')

## 2b. Upload Raw Dataset (Slow Path — Alternative)

Only use this if you don't have pre-computed mel spectrograms.
Training will be ~10× slower due to on-the-fly audio → mel conversion.

```bash
# Zip your local dataset:
#   cd dataset && tar -czf dataset.tar.gz dataset/
```

In [ ]:
import tarfile

if not USE_PRECOMPUTED:
    print('Uploading raw dataset...')
    uploaded_raw = files.upload()
    for fn in uploaded_raw.keys():
        if fn.endswith('.tar.gz') or fn.endswith('.tgz'):
            with tarfile.open(fn, 'r:gz') as tf:
                tf.extractall('.')
        elif fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zf:
                zf.extractall('.')

    # Find dataset directory
    data_dir = None
    for d in ['dataset/dataset', 'dataset']:
        if os.path.exists(d) and any(f.endswith('.wav') for f in glob.glob(f'{d}/**/*.wav', recursive=True)):
            data_dir = Path(d)
            break
    if data_dir is None:
        print('❌ Dataset not found. Check extraction.')
    else:
        solicited_dir = data_dir / 'raw_data' / 'solicited_data'
        wav_count = len(list(solicited_dir.glob('*.wav')))
        print(f'✅ Dataset at {data_dir} — {wav_count} WAV files')
else:
    print('Using pre-computed data (skipping raw upload).')
    data_dir = None

## 3. Load Dataset Metadata

Load the metadata mapping and explore the dataset.

In [ ]:
import pandas as pd
from pathlib import Path

if USE_PRECOMPUTED:
    # --- Load pre-computed metadata CSV ---
    df = pd.read_csv(csv_path)
    print(f'Loaded pre-computed metadata: {df.shape}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  TB+: {(df["tb_status"]==1).sum()} | TB-: {(df["tb_status"]==0).sum()}')
    print(f'  Unique patients: {df["participant"].nunique()}')

    # Verify some .npy files exist
    npy_exists = df['mel_path'].apply(os.path.exists).sum()
    print(f'  .npy files found: {npy_exists}/{len(df)}')
else:
    # --- Fallback: load from raw WAVs ---
    clinical_paths = list(data_dir.rglob('*Clinical_Meta_Info.csv'))
    solicited_paths = list(data_dir.rglob('*Solicited_Meta_Info.csv'))
    clinical = pd.read_csv(clinical_paths[0])
    solicited = pd.read_csv(solicited_paths[0])
    df = solicited.merge(clinical[['participant', 'tb_status']], on='participant')

    # Filter to only existing WAV files
    existing_fnames = set(f.name for f in solicited_dir.glob('*.wav'))
    df['_wav_exists'] = df['filename'].isin(existing_fnames)
    df = df[df['_wav_exists']].copy()
    df['mel_path'] = df['filename'].apply(lambda x: str(solicited_dir / x))

    print(f'Loaded raw metadata: {df.shape}')
    print(f'  TB+: {(df["tb_status"]==1).sum()} | TB-: {(df["tb_status"]==0).sum()}')
    print(f'  Unique patients: {df["participant"].nunique()}')

# Per-patient stats
patient_stats = df.groupby('participant').agg(
    n_files=('mel_path', 'count'),
    tb_status=('tb_status', 'first')
).reset_index()
print(f'\nPer-patient file count stats:')
print(patient_stats['n_files'].describe())

## 4. Model Architecture (MobileNetV4 + Res2TSM)

This is the same architecture from the original implementation.

In [ ]:
import torch
import torch.nn as nn
import timm

class TemporalShift(nn.Module):
    """Temporal shift module for temporal modeling."""
    def __init__(self, channels, shift_div=8):
        super().__init__()
        self.fold = channels // shift_div
    def forward(self, x):
        B, C, T, F = x.size()
        t = x.permute(0, 2, 1, 3).contiguous()
        out = torch.zeros_like(t)
        out[:, :-1, :self.fold, :] = t[:, 1:, :self.fold, :]
        out[:, 1:, self.fold:2*self.fold, :] = t[:, :-1, self.fold:2*self.fold, :]
        out[:, :, 2*self.fold:, :] = t[:, :, 2*self.fold:, :]
        return out.permute(0, 2, 1, 3)

class Res2TSMBlock(nn.Module):
    """Res2Net-style block with temporal shift."""
    def __init__(self, channels, scale=4, shift_div=8):
        super().__init__()
        self.scale = scale
        self.width = channels // scale
        self.temporal_shift = TemporalShift(channels, shift_div)
        self.convs = nn.ModuleList([
            nn.Conv2d(self.width, self.width, kernel_size=(3, 1),
                      padding=(1, 0), groups=self.width, bias=False)
            for _ in range(scale-1)
        ])
        self.bn = nn.BatchNorm2d(channels)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.temporal_shift(x)
        splits = torch.split(x, self.width, dim=1)
        y = splits[0]
        outs = [y]
        for i in range(1, self.scale):
            sp = splits[i] + y
            sp = self.convs[i-1](sp)
            y = sp
            outs.append(sp)
        out = torch.cat(outs, dim=1)
        return self.act(self.bn(out))

class MobileNetV4_Res2TSM(nn.Module):
    """
    MobileNetV4 backbone + Res2TSM temporal module + classification head.
    Input: (B, 3, 224, 224) mel spectrogram
    Output: (B,) sigmoid probability
    """
    def __init__(self, model_key='mobilenetv4_conv_blur_medium', scale=4, shift_div=8, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_key, pretrained=True, features_only=True)
        C = self.backbone.feature_info.channels()[-1]
        self.res2tsm = Res2TSMBlock(C, scale=scale, shift_div=shift_div)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(C, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        feat = self.backbone(x)[-1]  # (B, C, T, F)
        feat = self.res2tsm(feat)     # temporal modeling
        out = self.global_pool(feat).view(feat.size(0), -1)
        return self.fc(out).squeeze(1)

# Test forward pass
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_model = MobileNetV4_Res2TSM().to(DEVICE)
test_input = torch.randn(2, 3, 224, 224).to(DEVICE)
test_output = test_model(test_input)
print(f'Model loaded on {DEVICE}')
print(f'Input: {test_input.shape}')
print(f'Output: {test_output.shape}')
print(f'Params: {sum(p.numel() for p in test_model.parameters()):,}')
print(f'Trainable: {sum(p.numel() for p in test_model.parameters() if p.requires_grad):,}')

## 5. Data Pipeline

Loads pre-computed `.npy` mel spectrograms (fast) or raw WAV files (slow fallback).
Augmentation (SpecAugment + Gaussian noise) is applied **during training**.

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

# --- Augmentation (applied on pre-computed mels or raw mels) ---
class SpecAugment:
    """SpecAugment: time masking + frequency masking."""
    def __init__(self, freq_mask_max=15, time_mask_max=15, num_freq_masks=2, num_time_masks=2):
        self.freq_mask_max = freq_mask_max
        self.time_mask_max = time_mask_max
        self.num_freq_masks = num_freq_masks
        self.num_time_masks = num_time_masks

    def __call__(self, spec):
        # spec shape: (C, H, W) = (3, 224, 224)
        C, H, W = spec.shape
        for _ in range(self.num_freq_masks):
            f = np.random.randint(1, self.freq_mask_max + 1)
            f0 = np.random.randint(0, H - f)
            spec[:, f0:f0+f, :] = 0
        for _ in range(self.num_time_masks):
            t = np.random.randint(1, self.time_mask_max + 1)
            t0 = np.random.randint(0, W - t)
            spec[:, :, t0:t0+t] = 0
        return spec


def add_gaussian_noise(mel_rgb, noise_std=0.005):
    """Add mild Gaussian noise to mel spectrogram for augmentation."""
    return mel_rgb + np.random.randn(*mel_rgb.shape).astype(np.float32) * noise_std


# --- Dataset class (works with BOTH pre-computed .npy AND raw WAVs) ---
class CoughTBDataset(Dataset):
    """Dataset for CODA TB cough classification.

    Works in two modes:
    1. Pre-computed: 'mel_path' column points to .npy files
    2. Raw WAV: 'mel_path' column points to .wav files (uses load_and_segment + make_mel_rgb)
    """
    def __init__(self, dataframe, augment=False, use_precomputed=True):
        self.df = dataframe.reset_index(drop=True)
        self.augment = augment
        self.use_precomputed = use_precomputed
        self.spec_augment = SpecAugment() if augment else None

        if not use_precomputed:
            import librosa
            from scipy.ndimage import zoom

            self.SR = 16000
            self.CROP = 0.5
            self.N_MELS = 224
            self.FMAX = 8000
            self.TARGET_LEN = int(self.SR * self.CROP)

            self._librosa = librosa
            self._zoom = zoom

    def __len__(self):
        return len(self.df)

    def _load_mel_from_wav(self, wav_path):
        """Load audio and convert to mel spectrogram on-the-fly."""
        y, _ = self._librosa.load(wav_path, sr=self.SR)
        if len(y) == 0:
            return None
        y, _ = self._librosa.effects.trim(y, top_db=20)
        if len(y) >= self.TARGET_LEN:
            energy = np.convolve(y**2, np.ones(self.TARGET_LEN), mode='valid')
            start = np.argmax(energy)
            seg = y[start:start+self.TARGET_LEN]
        else:
            seg = np.zeros(self.TARGET_LEN, dtype=y.dtype)
            seg[:len(y)] = y

        mel = self._librosa.feature.melspectrogram(
            y=seg, sr=self.SR, n_mels=self.N_MELS, fmax=self.FMAX,
            hop_length=512, win_length=2048
        )
        mel_db = self._librosa.power_to_db(mel, ref=np.max)

        target_shape = (224, 224)
        if mel_db.shape != target_shape:
            zf = (target_shape[0] / mel_db.shape[0], target_shape[1] / mel_db.shape[1])
            resized = self._zoom(mel_db, zf, order=1)
        else:
            resized = mel_db

        if np.ptp(resized) > 0:
            normed = (resized - resized.min()) / np.ptp(resized)
        else:
            normed = np.zeros_like(resized)

        return np.stack([normed] * 3, axis=0).astype(np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel_path = row['mel_path']

        if self.use_precomputed:
            # Fast path: load pre-computed .npy file
            mel_rgb = np.load(mel_path).astype(np.float32)
        else:
            # Slow path: load WAV and compute mel on the fly
            mel_rgb = self._load_mel_from_wav(mel_path)
            if mel_rgb is None:
                return self.__getitem__((idx + 1) % len(self.df))

        # Data augmentation (applied regardless of path)
        if self.augment:
            mel_rgb = add_gaussian_noise(mel_rgb)
            if self.spec_augment:
                mel_rgb = self.spec_augment(mel_rgb)

        return (torch.from_numpy(mel_rgb).float(),
                torch.tensor(row['tb_status'], dtype=torch.float32))


print('✅ Data pipeline ready')
print(f'  Mode: {"Pre-computed .npy" if USE_PRECOMPUTED else "Raw WAV (slower)"}')

## 6. Train/Test Split (Patient-Level)

Stratified 80/20 split at the **patient level** to prevent data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.2
RANDOM_STATE = 42

# Patient-level split (stratified by tb_status)
unique_patients = df[['participant', 'tb_status']].drop_duplicates(subset=['participant'])
train_patients, test_patients = train_test_split(
    unique_patients,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=unique_patients['tb_status']
)

train_ids = set(train_patients['participant'])
test_ids = set(test_patients['participant'])

train_set = df[df['participant'].isin(train_ids)].copy()
test_set = df[df['participant'].isin(test_ids)].copy()

print(f"{'='*55}")
print(f'              DATASET SPLIT')
print(f"{'='*55}")
print(f"{'':<15} {'Patients':>10} {'Files':>10} {'TB+':>8} {'TB-':>8}")
print(f"{'-'*15} {'-'*10} {'-'*10} {'-'*8} {'-'*8}")
print(f"{'Train':<15} {len(train_ids):>10} {len(train_set):>10} {(train_set['tb_status']==1).sum():>8} {(train_set['tb_status']==0).sum():>8}")
print(f"{'Test':<15} {len(test_ids):>10} {len(test_set):>10} {(test_set['tb_status']==1).sum():>8} {(test_set['tb_status']==0).sum():>8}")
print(f"{'Total':<15} {len(unique_patients):>10} {len(df):>10} {(df['tb_status']==1).sum():>8} {(df['tb_status']==0).sum():>8}")
print(f"{'='*55}")

# Create datasets
BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = CoughTBDataset(train_set, augment=True, use_precomputed=USE_PRECOMPUTED)
test_dataset = CoughTBDataset(test_set, augment=False, use_precomputed=USE_PRECOMPUTED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f'\n✅ DataLoaders created')
print(f'   Train: {len(train_loader)} batches of {BATCH_SIZE}')
print(f'   Test:  {len(test_loader)} batches of {BATCH_SIZE}')

## 7. Training Utilities

In [ ]:
import time
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

class EarlyStopping:
    """Stop training when validation metric stops improving."""
    def __init__(self, patience=7, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0
        return self.early_stop


def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='Train', leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        all_preds.extend(outputs.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss, np.array(all_preds), np.array(all_labels)


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Evaluate on validation set."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='Val', leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        all_preds.extend(outputs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss, np.array(all_preds), np.array(all_labels)


def compute_metrics(y_true, y_pred_probs, threshold=0.5):
    """Compute classification metrics."""
    y_pred = (y_pred_probs > threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        'sensitivity': sensitivity,
        'specificity': specificity,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_pred_probs) if len(np.unique(y_true)) > 1 else 0.0,
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    }
    return metrics


def print_metrics(metrics, prefix=''):
    """Print metrics in a formatted way."""
    print(f'{prefix} Sensitivity: {metrics["sensitivity"]:.3f}  ({metrics["sensitivity"]*100:.1f}%)')
    print(f'{prefix} Specificity: {metrics["specificity"]:.3f}  ({metrics["specificity"]*100:.1f}%)')
    print(f'{prefix} Accuracy:    {metrics["accuracy"]:.3f}  ({metrics["accuracy"]*100:.1f}%)')
    print(f'{prefix} Precision:   {metrics["precision"]:.3f}')
    print(f'{prefix} F1 Score:    {metrics["f1"]:.3f}')
    print(f'{prefix} ROC-AUC:     {metrics["roc_auc"]:.3f}')
    print(f'{prefix} Confusion:   TN={metrics["confusion_matrix"]["tn"]} FP={metrics["confusion_matrix"]["fp"]} '
          f'FN={metrics["confusion_matrix"]["fn"]} TP={metrics["confusion_matrix"]["tp"]}')


print('✅ Training utilities ready')

## 8. Phase 1: Train Head + Res2TSM (Backbone Frozen)

Train only the classification head and Res2TSM block for 10 epochs.
The MobileNetV4 backbone (pretrained on ImageNet) is frozen.

In [ ]:
EPOCHS_PHASE1 = 10
LR_PHASE1 = 1e-3

# Reset model
model = MobileNetV4_Res2TSM('mobilenetv4_conv_blur_medium').to(DEVICE)

# Freeze backbone
for param in model.backbone.parameters():
    param.requires_grad = False

# Trainable params check
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}  ({trainable/total*100:.1f}%)')
print(f'  Frozen backbone, training head + Res2TSM only')

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE1, weight_decay=1e-4
)

# Use AMP for speed if available
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

print(f"\n{'='*55}")
print(f'  PHASE 1: Frozen Backbone Training')
print(f'  Epochs: {EPOCHS_PHASE1} | LR: {LR_PHASE1} | Optimizer: AdamW')
print(f"{'='*55}")

history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_sens': [], 'val_spec': []}
best_val_auc = 0.0

for epoch in range(1, EPOCHS_PHASE1 + 1):
    print(f'\n--- Epoch {epoch}/{EPOCHS_PHASE1} ---')

    train_loss, train_preds, train_labels = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_preds, val_labels = validate(model, test_loader, criterion, DEVICE)

    train_metrics = compute_metrics(train_labels, train_preds)
    val_metrics = compute_metrics(val_labels, val_preds)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_metrics['roc_auc'])
    history['val_sens'].append(val_metrics['sensitivity'])
    history['val_spec'].append(val_metrics['specificity'])

    print(f'  Train Loss: {train_loss:.4f} | AUC: {train_metrics["roc_auc"]:.3f}')
    print(f'  Val   Loss: {val_loss:.4f} | AUC: {val_metrics["roc_auc"]:.3f}')
    print_metrics(val_metrics, prefix='  Val  ')

    if val_metrics['roc_auc'] > best_val_auc:
        best_val_auc = val_metrics['roc_auc']
        torch.save(model.state_dict(), 'best_phase1.pth')
        print(f'  🏆 New best model saved! (AUC: {best_val_auc:.3f})')

print(f"\n{'='*55}")
print(f'  PHASE 1 COMPLETE')
print(f'  Best Val AUC: {best_val_auc:.3f}')
print(f"{'='*55}")

## 9. Phase 2: Full Fine-Tuning

Unfreeze backbone and fine-tune all parameters with cosine annealing LR schedule.

In [ ]:
EPOCHS_PHASE2 = 40
LR_PHASE2 = 1e-4
EARLY_STOPPING_PATIENCE = 7

# Load best Phase 1 weights
if os.path.exists('best_phase1.pth'):
    model.load_state_dict(torch.load('best_phase1.pth'))
    print('✅ Loaded best Phase 1 weights')

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,} / {sum(p.numel() for p in model.parameters()):,} ({trainable/total*100:.1f}%)')

# Optimizer with cosine annealing
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PHASE2, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE2)

early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, min_delta=0.001)

print(f"\n{'='*55}")
print(f'  PHASE 2: Full Fine-Tuning')
print(f'  Epochs: {EPOCHS_PHASE2} | LR: {LR_PHASE2} (cosine) | Patience: {EARLY_STOPPING_PATIENCE}')
print(f"{'='*55}")

best_val_auc = 0.0

for epoch in range(1, EPOCHS_PHASE2 + 1):
    print(f'\n--- Epoch {epoch}/{EPOCHS_PHASE2} (LR: {scheduler.get_last_lr()[0]:.2e}) ---')

    train_loss, train_preds, train_labels = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_preds, val_labels = validate(model, test_loader, criterion, DEVICE)

    scheduler.step()

    train_metrics = compute_metrics(train_labels, train_preds)
    val_metrics = compute_metrics(val_labels, val_preds)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_metrics['roc_auc'])
    history['val_sens'].append(val_metrics['sensitivity'])
    history['val_spec'].append(val_metrics['specificity'])

    print(f'  Train Loss: {train_loss:.4f} | AUC: {train_metrics["roc_auc"]:.3f}')
    print(f'  Val   Loss: {val_loss:.4f} | AUC: {val_metrics["roc_auc"]:.3f}')
    print_metrics(val_metrics, prefix='  Val  ')

    # Save best model
    if val_metrics['roc_auc'] > best_val_auc:
        best_val_auc = val_metrics['roc_auc']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': best_val_auc,
            'val_metrics': val_metrics
        }, 'best_model.pth')
        print(f'  🏆 New best model saved! (AUC: {best_val_auc:.3f})')

    # Early stopping
    if early_stopping(val_metrics['roc_auc']):
        print(f'\n🛑 Early stopping triggered after {epoch} epochs')
        break

print(f"\n{'='*55}")
print(f'  PHASE 2 COMPLETE')
print(f'  Best Val AUC: {best_val_auc:.3f}')
print(f"{'='*55}")

## 10. Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].axvline(x=EPOCHS_PHASE1 - 1, color='gray', linestyle='--', alpha=0.5, label='Phase 1→2')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# AUC
axes[1].plot(history['val_auc'], label='Val AUC', color='green', linewidth=2)
axes[1].axvline(x=EPOCHS_PHASE1 - 1, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('Validation ROC-AUC')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Sensitivity & Specificity
axes[2].plot(history['val_sens'], label='Sensitivity', linewidth=2)
axes[2].plot(history['val_spec'], label='Specificity', linewidth=2)
axes[2].axvline(x=EPOCHS_PHASE1 - 1, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Score')
axes[2].set_title('Sensitivity & Specificity')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training history saved: training_history.png')

## 11. Final Evaluation (Per-Patient)

Aggregate predictions per patient and compute metrics at patient level.

In [ ]:
# Load best model
if os.path.exists('best_model.pth'):
    checkpoint = torch.load('best_model.pth', map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'✅ Loaded best model from epoch {checkpoint["epoch"]} (Val AUC: {checkpoint["val_auc"]:.3f})')

model.eval()

# Run inference on test set
eval_dataset = CoughTBDataset(test_set, augment=False, use_precomputed=USE_PRECOMPUTED)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

with torch.no_grad():
    all_probs = []
    all_labels = []
    all_participants = []
    for inputs, labels in tqdm(eval_loader, desc='Inference'):
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        all_probs.extend(outputs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_set = test_set.copy()
test_set['pred_prob'] = all_probs

# Per-patient aggregation
patient_results = test_set.groupby('participant').agg(
    mean_prob=('pred_prob', 'mean'),
    tb_status=('tb_status', 'first'),
    n_coughs=('mel_path', 'count')
).reset_index()

patient_results['pred'] = (patient_results['mean_prob'] > 0.5).astype(int)

p_metrics = compute_metrics(patient_results['tb_status'].values, patient_results['mean_prob'].values)

print(f"\n{'='*55}")
print(f'  FINAL EVALUATION — Per-Patient Level')
print(f'  Patients: {len(patient_results)}')
print(f"{'='*55}")
print_metrics(p_metrics)
print(f"{'='*55}")

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(patient_results['tb_status'], patient_results['mean_prob'])
youden_idx = np.argmax(tpr - fpr)
best_thresh = thresholds[youden_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ROC
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {p_metrics["roc_auc"]:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax1.plot(fpr[youden_idx], tpr[youden_idx], 'ro', markersize=8,
         label=f'Youden: thresh={best_thresh:.2f}')
ax1.set_xlabel('1 - Specificity (FPR)')
ax1.set_ylabel('Sensitivity (TPR)')
ax1.set_title('ROC Curve — CoughTB (Our Model)')
ax1.legend(loc='lower right')
ax1.grid(alpha=0.3)

# Confusion Matrix
cm = np.array([[p_metrics['confusion_matrix']['tn'], p_metrics['confusion_matrix']['fp']],
               [p_metrics['confusion_matrix']['fn'], p_metrics['confusion_matrix']['tp']]])
im = ax2.imshow(cm, cmap='Blues')
ax2.set_xticks([0, 1])
ax2.set_yticks([0, 1])
ax2.set_xticklabels(['Pred Neg', 'Pred Pos'])
ax2.set_yticklabels(['True Neg', 'True Pos'])
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
ax2.set_title('Confusion Matrix (Patient-Level)')

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nYouden's J optimal threshold: {best_thresh:.3f}")
print(f'  At threshold: Sensitivity={tpr[youden_idx]:.3f}, Specificity={1-fpr[youden_idx]:.3f}')
print('\n✅ Evaluation figures saved')

## 12. Export Model Weights

Save the final model in a format compatible with the existing web app.

In [ ]:
# Save final model
final_path = 'cough_tb_model.pth'
torch.save(model.state_dict(), final_path)
print(f'✅ Model saved: {final_path}')
print(f'   Size: {os.path.getsize(final_path) / 1024 / 1024:.2f} MB')

# Also save as checkpoint with metadata
checkpoint_path = 'cough_tb_checkpoint.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'MobileNetV4_Res2TSM',
    'model_key': 'mobilenetv4_conv_blur_medium',
    'scale': 4,
    'shift_div': 8,
    'dropout': 0.3,
    'val_auc': p_metrics['roc_auc'],
    'val_sensitivity': p_metrics['sensitivity'],
    'val_specificity': p_metrics['specificity'],
    'input_shape': (3, 224, 224),
    'sr': 16000,
    'crop_seconds': 0.5,
    'n_mels': 224,
    'fmax': 8000
}, checkpoint_path)
print(f'✅ Checkpoint saved: {checkpoint_path}')
print(f'   Size: {os.path.getsize(checkpoint_path) / 1024 / 1024:.2f} MB')

In [ ]:
# Download the model
from google.colab import files

print('Download the trained model:')
files.download('cough_tb_model.pth')

print('\nTo use with web app:')
print('1. Download cough_tb_model.pth')
print('2. Rename to model.pth')
print('3. Place in the web/ directory (replacing the old model.pth)')

## 13. Test with Sample Audio

Upload a cough sound to test the trained model.

In [ ]:
import librosa
import IPython.display as ipd
from google.colab import files
from scipy.ndimage import zoom
import numpy as np

SR = 16000
CROP = 0.5
TARGET_LEN = int(SR * CROP)

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n{'='*55}")
    print(f'FILE: {filename}')
    print(f"{'='*55}")

    ipd.display(ipd.Audio(filename))

    # Load and segment
    y, _ = librosa.load(filename, sr=SR)
    if len(y) > 0:
        y, _ = librosa.effects.trim(y, top_db=20)
        if len(y) >= TARGET_LEN:
            energy = np.convolve(y**2, np.ones(TARGET_LEN), mode='valid')
            start = np.argmax(energy)
            y_seg = y[start:start+TARGET_LEN]
        else:
            y_seg = np.zeros(TARGET_LEN, dtype=y.dtype)
            y_seg[:len(y)] = y
    else:
        print('❌ Cannot process audio')
        continue

    # Mel spectrogram
    mel = librosa.feature.melspectrogram(y=y_seg, sr=SR, n_mels=224, fmax=8000,
                                          hop_length=512, win_length=2048)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    if mel_db.shape != (224, 224):
        zf = (224/mel_db.shape[0], 224/mel_db.shape[1])
        mel_db = zoom(mel_db, zf, order=1)
    normed = ((mel_db - mel_db.min()) / np.ptp(mel_db)) if np.ptp(mel_db) > 0 else np.zeros_like(mel_db)
    mel_rgb = np.stack([normed] * 3, axis=0).astype(np.float32)

    with torch.no_grad():
        input_tensor = torch.from_numpy(mel_rgb).float().unsqueeze(0).to(DEVICE)
        prob = model(input_tensor).cpu().numpy()[0]

    if prob > 0.5:
        print(f'\n  ⚠️  TB DETECTED  (confidence: {prob*100:.1f}%)')
    else:
        print(f'\n  ✅ No TB detected  (confidence: {(1-prob)*100:.1f}%)')
    print(f'  TB Probability: {prob:.4f}')

---
## Summary

✅ **Training Complete!** Here's what was accomplished:

1. **Phase 1** (10 epochs): Trained head + Res2TSM with frozen ImageNet backbone
2. **Phase 2** (up to 40 epochs): Full fine-tuning with cosine annealing LR
3. **Early stopping**: patience = 7
4. **Augmentation**: SpecAugment (time/freq masking) + Gaussian noise
5. **Data split**: Patient-level 80/20 stratified
6. **Data mode**: Uses pre-computed .npy (fast) or raw WAV (slow fallback)

### Files saved:
- `best_phase1.pth` — best model after Phase 1
- `best_model.pth` — best overall checkpoint (with metadata)
- `cough_tb_model.pth` — final model weights (for web app)
- `training_history.png` — loss/AUC/sens/spec curves
- `evaluation_results.png` — ROC curve + confusion matrix

### Next Steps
1. Download `cough_tb_model.pth` → rename to `model.pth` → replace in `web/`
2. Update the paper with your training results
3. Test on external datasets (TBscreen, etc.)
4. Export to ONNX/TFLite for mobile deployment